In [ ]:
# ==============================================================================
# Fourier Transform Method: Output of an LTI System
# Given:
# h[n] = 5(-1/2)^n u[n]
# x[n] = (1/3)^n u[n]
# Target Publication: Springer
# ==============================================================================

import sympy as sp
from IPython.display import display, Markdown

sp.init_printing(use_unicode=True)

# ------------------------------------------------------------------------------
# 1. Symbolic Definitions
# ------------------------------------------------------------------------------

omega = sp.Symbol('omega', real=True)
n = sp.Symbol('n', integer=True, nonnegative=True)
q = sp.Symbol('q')

H = sp.Function('H')
X = sp.Function('X')
Y = sp.Function('Y')
y = sp.Function('y')
u = sp.Function('u')

# Given impulse response: h[n] = A_h * a_h^n * u[n]
A_h = sp.Rational(5, 1)
a_h = -sp.Rational(1, 2)

# Given input: x[n] = A_x * a_x^n * u[n]
A_x = sp.Rational(1, 1)
a_x = sp.Rational(1, 3)

# ------------------------------------------------------------------------------
# 2. Fourier Transform of the Impulse Response
#
# F{A*a^n*u[n]} = A/(1-a*e^(-jω))
# ------------------------------------------------------------------------------

H_q = sp.simplify(A_h / (1 - a_h * q))
H_omega = sp.simplify(H_q.subs(q, sp.exp(-sp.I * omega)))

display(Markdown("### **Fourier Transform of the Impulse Response**"))
display(sp.Eq(H(sp.exp(sp.I * omega)), H_omega))

# ------------------------------------------------------------------------------
# 3. Fourier Transform of the Input
# ------------------------------------------------------------------------------

X_q = sp.simplify(A_x / (1 - a_x * q))
X_omega = sp.simplify(X_q.subs(q, sp.exp(-sp.I * omega)))

display(Markdown("### **Fourier Transform of the Input**"))
display(sp.Eq(X(sp.exp(sp.I * omega)), X_omega))

# ------------------------------------------------------------------------------
# 4. Fourier Transform of the Output
#
# Y(e^jw) = H(e^jw) X(e^jw)
# ------------------------------------------------------------------------------

Y_q = sp.cancel(H_q * X_q)
Y_omega = sp.simplify(Y_q.subs(q, sp.exp(-sp.I * omega)))

display(Markdown("### **Fourier Transform of the Output**"))
display(sp.Eq(Y(sp.exp(sp.I * omega)), Y_omega))

# ------------------------------------------------------------------------------
# 5. Symbolic Partial Fraction Decomposition
#
# The decomposition is obtained automatically from Y(q).
# No result from the textbook solution is supplied here.
# ------------------------------------------------------------------------------

denominator = sp.denom(Y_q)
poles_q = sp.solve(sp.Eq(denominator, 0), q)

partial_terms = []

for r in poles_q:
    p = sp.simplify(1 / r)
    A = sp.simplify(-p * sp.residue(Y_q, q, r))
    partial_terms.append((A, p))

# Construct the two partial-fraction terms explicitly
partial_fraction_terms = [A / (1 - p * sp.exp(-sp.I * omega)) for A, p in partial_terms]
Y_partial_omega = sp.Add(*partial_fraction_terms)

display(Markdown("### **Partial Fraction Expansion**"))
display(sp.Eq(Y(sp.exp(sp.I * omega)), Y_partial_omega))

# ------------------------------------------------------------------------------
# 6. Automatic Inverse Fourier Transform
#
# A/(1-p*e^(-jω))  <-->  A*p^n*u[n]
# ------------------------------------------------------------------------------

y_without_u = sp.simplify(sum(A * p**n for A, p in partial_terms))
y_symbolic = sp.factor(y_without_u)

display(Markdown("### **Output Signal**"))
display(sp.Eq(y(n), y_symbolic * u(n)))